# Incompressible LES: Pitz Daily with Smagorinsky

## Preamble

In [1]:
# Standard Library
from pathlib import Path
import foamnordic as fno
import hedgehogs as hdg

### Directories

In [ ]:
# Case Type and Name
CASE_TYPE = "les"
CASE_NAME = "pitzDailySmag"
CATEGORY = "incompressible"

# FoamNordic Project Directory
PROJECT_DIR = Path.cwd().resolve()
BASE_DIR = next((path for path in (PROJECT_DIR, *PROJECT_DIR.parents) if (path / "tutorials" / "openfoam_tutorials").is_dir()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Run this notebook from inside the FoamNordic source tree")
MAIN_DIR = BASE_DIR / "tutorials" / "foamnordic_tutorials" / CATEGORY
OF_SCRIPT_DIR = BASE_DIR / "tutorials" / "openfoam_tutorials" / CATEGORY / CASE_TYPE / CASE_NAME

# Output Directory
MODEL_DIR = MAIN_DIR / "model"
OUTPUT_DIR = MAIN_DIR / "output"

for directory in [MODEL_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

### Configuration

In [ ]:
# Hedgehogs configuration
hdg.set_style()

In [ ]:
# HPC configuration
ACCOUNT = "<allocation-account>"  # e.g. project_xxxxxxx
PARTITION = "small"
TIME = "00:15:00"
N_NODES = 1
N_TASKS = 1
CPUS_PER_TASK = 1
MEM_PER_CPU = "2G"

# FoamNordic Slurm configuration
of_scheduler = fno.Slurm.openfoam(
    nodes=N_NODES,
    ntasks=N_TASKS,
    cpus_per_task=CPUS_PER_TASK,
    mem_per_cpu=MEM_PER_CPU,
)

model_scheduler = fno.Slurm.model(
    cpus_per_task=1,
    mem_per_cpu=MEM_PER_CPU
)

scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time=TIME,
    openfoam=of_scheduler,
    model=model_scheduler
)

## Example - Closure Modelling

### Smagorinsky Closure

In [ ]:
# Smagorinsky model coefficients
C_K = 0.0265463553
C_E = 1.048

In [ ]:
# Smagorinsky function for subgrid-scale turbulence modelling
def smagorinsky_function(velocity_grad, delta, C_k=C_K, C_e=C_E):
    strain_rate = fno.Math.symm(value=velocity_grad)
    strain_rate_trace = fno.Math.trace(value=strain_rate)
    dev_strain_rate = fno.Math.dev(value=strain_rate)

    coefficient_a = C_e / delta
    coefficient_b = (2.0 / 3.0) * strain_rate_trace
    coefficient_c = 2.0 * C_k * delta * fno.Math.ddot(left=dev_strain_rate, right=strain_rate)

    discriminant = coefficient_b**2 + 4.0 * coefficient_a * coefficient_c
    sqrt_k = (-coefficient_b + fno.Math.sqrt(value=fno.Math.maximum(left=discriminant, right=0.0))) / (2.0 * coefficient_a)
    sqrt_k = fno.Math.maximum(left=sqrt_k, right=0.0)

    eddy_viscosity = C_k * delta * sqrt_k

    return eddy_viscosity

In [ ]:
# Define the Smagorinsky closure
smagorinsky_closure = fno.Closure(
    name="nutFjord",
    operator=fno.Operator.function(function=smagorinsky_function),
    inputs={
        "velocity_grad": fno.Field.grad(value="U"),
        "delta": fno.Field.delta(),
    },
    outputs={
        "eddy_viscosity": fno.Field(name="nut"),
    },
    key=None,
)

### Case Definition

In [ ]:
# Initialize the OpenFOAM case
case = fno.OpenFOAM.Case(
    name=CASE_NAME,
    case_dir=OF_SCRIPT_DIR,
    run_dir=OUTPUT_DIR,
    of_cmd="module load openfoam/2512",
    shell="bash",
    application="pimpleFoam",
    ranks=N_TASKS,
)

case.initialize(ranks=N_TASKS, mesh="blockMesh", validate_mesh=True);

### Submit Job

In [8]:
# Connect the OpenFOAM case and execution resources
longship = fno.Longship(
    case=case,
    closures=(smagorinsky_closure,),
    scheduler=scheduler,
    verbose=True,
)

Resource,OpenFOAM,Model,Allocation
Processes,1 tasks,1 host(s),1 node(s)
CPU,1 cores,1 cores,2 cores
Memory,2 GiB,2 GiB,4.03125 GiB
Field programs,-,1,1
SHM,-,~32 MiB,~32 MiB


In [9]:
# Launch the workload
run = longship.launch(verbose=None)

[FoamNordic] nutFjord execution lane: native OpenFOAM.
[FoamNordic] Preparing mesh with blockMesh: pitzDailySmag
[FoamNordic] Mesh is ready: pitzDailySmag
[FoamNordic] Sailing submitted with Job ID: 1289867
[FoamNordic] Sailing has launched with Job ID: 1289867
[FoamNordic] Sailing started at: 2026-09-13T14:38:59
[FoamNordic] Sailing in background: pitzDailySmag


In [10]:
# Wait for the job to complete (polling the job status)
result = run.stop(force=False, timeout=3600, progress=True)

In [11]:
# Summary of the job result
result.summary(style="compact", display=True);

Job ID,Name,Status,Partition,Node,Elapsed
1289867,pitzDailySmag,succeeded,small,rc5130,00:01:03


### Post-processing

In [12]:
# Postprocessing
post = result.postprocess

velocity = post.field(name="U", time_idx=-1, physical_time=None)
pressure = post.field(name="p", time_idx=-1, physical_time=None)

print("U shape:", velocity.shape)
print("p shape:", pressure.shape)

statistics = post.statistics(
    fields=["U", "p", "nut"],
    time_idx=-1,
    physical_time=None,
    verbose=True,
)

U shape: (12225, 3)
p shape: (12225,)


Field,Min,Max,Mean,Std,RMS
U,2.180695e-02,1.395087e+01,6.371092e+00,2.649377e+00,6.900001e+00
p,-3.573670e+01,1.143900e+02,6.502952e+01,2.183933e+01,6.859880e+01
nut,2.761320e-10,1.352340e-04,2.312463e-06,5.793488e-06,6.237947e-06
